# CNV and its association with target-locked P3a

This notebook performs the exploratory CNV analyses: probability effects, associations with self-functioning impairment, measurement reliability, and trial-level CNV–P3a coupling. It prepares models M3–M6, Table 6, and Figures 4–5.

## Inputs and execution

- `eeg_features.xlsx` and the `evoked` folder from `02_eeg_features.ipynb`.
- `erp_behavior_statistics.xlsx` from `03_erp_behavior_statistics.ipynb`, containing the participant scores and the hash of the EEG input.

Set the paths below, change `RUN_ANALYSES` to `True`, restart the kernel, and run all cells. Inspect sample sizes, score scaling, convergence, and warnings before exporting. With `RUN_ANALYSES = False`, no study data are read or analyzed.

Dependencies: `numpy`, `pandas`, `scipy`, `statsmodels`, `matplotlib`, `mne`, `openpyxl`, and Jupyter.

In [1]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import platform
import warnings

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import mne
from IPython.display import Markdown, display

software_versions = pd.DataFrame(
    [("Python", platform.python_version())] + [(name, version(name)) for name in
     ("numpy", "pandas", "scipy", "statsmodels", "matplotlib", "mne", "openpyxl")],
    columns=["software", "version"],
)
pd.set_option("display.max_rows", 25)
pd.set_option("display.max_columns", 16)
display(software_versions)

,software,version
0,Python,3.12.14
1,numpy,2.3.5
2,pandas,2.2.3
3,scipy,1.17.0
4,statsmodels,0.15.0
5,matplotlib,3.10.8
6,mne,1.13.2
7,openpyxl,3.1.5


In [2]:
RUN_ANALYSES = False
EEG_FEATURES_FILE = Path("PATH_TO_EEG_OUTPUTS/eeg_features.xlsx").expanduser()
ERP_STATISTICS_FILE = Path("PATH_TO_STATISTICAL_OUTPUTS/erp_behavior_statistics.xlsx").expanduser()
EVOKED_DIR = Path("PATH_TO_EEG_OUTPUTS/evoked").expanduser()
OUTPUT_DIR = Path("PATH_TO_CNV_STATISTICAL_OUTPUTS").expanduser()

EXPECTED_N_ERP_PARTICIPANTS = 69
MIN_CNV_TRIALS_PER_CONTEXT = 30
N_RELIABILITY_SPLITS = 2_000
RELIABILITY_SEED = 21
CNV_CHANNEL = "Fz"
CNV_WINDOW = (0.350, 1.000)

MAKE_FIGURES = True
EXPORT_OUTPUTS = False
OVERWRITE_OUTPUTS = False

if EXPORT_OUTPUTS and not RUN_ANALYSES:
    raise ValueError("Enable RUN_ANALYSES before exporting study results.")
print("Mode:", "study analysis" if RUN_ANALYSES else "no study data loaded")

Mode: no study data loaded


## Validate trial selection and pairing

CNV selection is performed in notebook 02: retained EEG epochs must have a detected saccade with latency ≥0 ms relative to the target, regardless of its direction. Participants must have at least 30 retained trials in each probability context. This notebook verifies these conditions without applying an additional latency threshold.

M3 uses every retained CNV trial; M4 uses trials from participants with available self-functioning and age scores. Coupling models begin with the complete intersection of retained CNV and P3a trials, matched by participant and target sample, and use model-specific complete cases. The exported paired table is checked against that intersection. Figure 4 uses the full CNV cohort; Figure 5 uses the M6 model.

Self-functioning and age scores are loaded from the `Participant_scores` sheet of notebook 03. They are not restandardized on the CNV subset. The EEG workbook must match the file used in notebook 03; its SHA-256 hash is verified.

In [3]:
def normalize_subjects(frame):
    frame = frame.copy()
    if "subject" not in frame or frame["subject"].isna().any():
        raise ValueError("Nonmissing participant identifiers are required.")
    frame["subject"] = frame["subject"].astype(str).str.strip().str.upper()
    if frame["subject"].eq("").any():
        raise ValueError("Participant identifiers must not be blank.")
    return frame


def validate_trial_table(frame, name, amplitudes):
    required = {"subject", "trial", "target_sample", "probability", "validity", "condition", *amplitudes}
    if missing := required - set(frame):
        raise ValueError(f"{name}: missing fields {sorted(missing)}.")
    frame = normalize_subjects(frame)
    if frame.empty or frame.duplicated(["subject", "target_sample"]).any():
        raise ValueError(f"{name}: trials must be nonempty and unique by participant/target sample.")
    if frame.duplicated(["subject", "trial"]).any():
        raise ValueError(f"{name}: duplicate participant/trial records.")
    for column in ["trial", "target_sample", "probability", *amplitudes]:
        frame[column] = pd.to_numeric(frame[column], errors="raise")
        if not np.isfinite(frame[column]).all():
            raise ValueError(f"{name}: nonfinite values in {column}.")
    for column in ["trial", "target_sample"]:
        if not frame[column].eq(np.floor(frame[column])).all():
            raise ValueError(f"{name}: {column} must contain integer indices.")
    if not frame["probability"].isin([50, 80]).all() or not frame["validity"].isin(["valid", "invalid"]).all():
        raise ValueError(f"{name}: unknown experimental condition.")
    expected = frame["validity"] + "_" + frame["probability"].astype(int).astype(str)
    if not frame["condition"].eq(expected).all():
        raise ValueError(f"{name}: inconsistent condition labels.")
    return frame.sort_values(["subject", "target_sample"], kind="stable").reset_index(drop=True)


def load_cnv_inputs():
    eeg_hash = hashlib.sha256(EEG_FEATURES_FILE.read_bytes()).hexdigest()
    source_files = pd.read_excel(ERP_STATISTICS_FILE, sheet_name="Input_files", engine="openpyxl")
    if "sha256" not in source_files or eeg_hash not in set(source_files["sha256"].astype(str)):
        raise ValueError("The EEG workbook differs from the input recorded by notebook 03. Run 03 with this EEG file first.")
    sheets = pd.read_excel(EEG_FEATURES_FILE,
        sheet_name=["ERP_trials", "CNV_trials", "ERP_CNV_pairs", "CNV_participant_QC"], engine="openpyxl")
    erp = validate_trial_table(sheets["ERP_trials"], "ERP_trials", ["erp_amplitude_uv"])
    cnv = validate_trial_table(sheets["CNV_trials"], "CNV_trials", ["cnv_amplitude_uv"])
    pairs = validate_trial_table(sheets["ERP_CNV_pairs"], "ERP_CNV_pairs", ["erp_amplitude_uv", "cnv_amplitude_uv"])
    scores = normalize_subjects(pd.read_excel(ERP_STATISTICS_FILE, sheet_name="Participant_scores", engine="openpyxl"))
    if {"self_z", "age_z"} - set(scores) or scores["subject"].duplicated().any():
        raise ValueError("Participant_scores must have one row per participant, with self_z and age_z.")
    if set(scores["subject"]) != set(erp["subject"]):
        raise ValueError("The ERP cohort and Participant_scores do not match.")
    if len(scores) != EXPECTED_N_ERP_PARTICIPANTS:
        raise ValueError("The ERP cohort differs from EXPECTED_N_ERP_PARTICIPANTS.")
    for column in ["self_z", "age_z"]:
        scores[column] = pd.to_numeric(scores[column], errors="raise")
        if np.isinf(scores[column]).any():
            raise ValueError(f"Infinite values in {column}.")
    if not set(cnv["subject"]) <= set(erp["subject"]):
        raise ValueError("The CNV cohort must be a subset of the ERP cohort.")
    counts = cnv.groupby(["subject", "probability"]).size().unstack(fill_value=0).reindex(columns=[50, 80], fill_value=0)
    if counts.lt(MIN_CNV_TRIALS_PER_CONTEXT).any(axis=None):
        raise ValueError("CNV trials do not meet the configured minimum in both probability contexts.")
    if "latency_ms" not in cnv or "saccade_detected" not in cnv:
        raise ValueError("CNV_trials must retain the reviewed latency and detection fields from notebook 02.")
    latency = pd.to_numeric(cnv["latency_ms"], errors="raise")
    if not cnv["saccade_detected"].eq(True).all() or not np.isfinite(latency).all() or not latency.ge(0).all():
        raise ValueError("CNV_trials contain an undetected or pretarget saccade.")
    qc = normalize_subjects(sheets["CNV_participant_QC"])
    if "included" not in qc or qc["subject"].duplicated().any() or set(qc["subject"]) != set(erp["subject"]):
        raise ValueError("CNV_participant_QC must describe every ERP participant once.")
    if not qc["included"].isin([True, False]).all():
        raise ValueError("The CNV inclusion flags must be Boolean.")
    if set(qc.loc[qc["included"].eq(True), "subject"]) != set(cnv["subject"]):
        raise ValueError("CNV inclusion flags disagree with the trial table.")

    keys = ["subject", "target_sample"]
    common = erp.merge(cnv[keys + ["trial", "probability", "validity", "cnv_amplitude_uv"]],
                      on=keys, how="inner", validate="one_to_one", suffixes=("", "_cnv"))
    for column in ["trial", "probability", "validity"]:
        if not common[column].eq(common[column + "_cnv"]).all():
            raise ValueError(f"P3a and CNV disagree on {column} for a matched target.")
    common = common.sort_values(keys).reset_index(drop=True)
    actual = pairs.sort_values(keys).reset_index(drop=True)
    if len(common) != len(actual) or not common[keys].equals(actual[keys]):
        raise ValueError("ERP_CNV_pairs is not the complete one-to-one intersection of ERP and CNV trials.")
    for column in ["erp_amplitude_uv", "cnv_amplitude_uv"]:
        if not np.allclose(common[column], actual[column], rtol=1e-10, atol=1e-10):
            raise ValueError(f"Paired amplitudes disagree with the source trial table: {column}.")
    for column in ["trial", "probability", "validity", "condition"]:
        if not common[column].eq(actual[column]).all():
            raise ValueError(f"Paired trial metadata disagree: {column}.")
    if set(pairs["subject"]) != set(cnv["subject"]):
        raise ValueError("At least one CNV participant has no matched P3a trials.")
    input_files = pd.DataFrame([
        {"role": "EEG features", "file": EEG_FEATURES_FILE.name, "sha256": eeg_hash},
        {"role": "ERP statistics and scores", "file": ERP_STATISTICS_FILE.name,
         "sha256": hashlib.sha256(ERP_STATISTICS_FILE.read_bytes()).hexdigest()},
    ])
    return cnv, pairs, scores.sort_values("subject").reset_index(drop=True), qc, input_files

## CNV summaries and measurement reliability

Participant CNV level is the equally weighted mean of the 80% and 50% condition averages. The descriptive probability contrast is `CNV50 − CNV80`: positive values indicate a more negative CNV in the 80% context. The probability coefficient in M3/M4 has the opposite subtraction order, `CNV80 − CNV50`.

For each of 2,000 random splits (seed 21), trials are permuted separately within participant and probability context. The first half contains `floor(n/2)` trials and the second contains the remainder. Pearson correlations across participants are calculated between halves for the probability contrast and the mean CNV level. The Spearman–Brown formula `2r/(1+r)` is applied to each split, then averaged across splits. Negative estimates are retained; undefined estimates are counted separately.

The 2.5th and 97.5th percentiles describe variability across random splits; they are not participant-bootstrap confidence intervals. Standardized measurement error (SME) for each participant and context is trial SD/√n; RMS SME is the root mean square of those values across participants, in microvolts.

In [4]:
def summarize_cnv_trials(cnv_trials):
    groups = cnv_trials.groupby(["subject", "probability"])["cnv_amplitude_uv"]
    condition = groups.agg(n_trials="size", mean_cnv_uv="mean", sd_cnv_uv="std").reset_index()
    condition["sme_uv"] = condition["sd_cnv_uv"] / np.sqrt(condition["n_trials"])
    means = condition.pivot(index="subject", columns="probability", values="mean_cnv_uv")
    counts = condition.pivot(index="subject", columns="probability", values="n_trials")
    participant = pd.DataFrame({"cnv_80_uv": means[80], "cnv_50_uv": means[50],
        "n_80": counts[80], "n_50": counts[50]})
    participant["probability_negativity_uv"] = participant["cnv_50_uv"] - participant["cnv_80_uv"]
    participant["mean_cnv_uv"] = (participant["cnv_50_uv"] + participant["cnv_80_uv"]) / 2
    summaries = []
    for probability, frame in condition.groupby("probability"):
        summaries.append({"probability": probability, "n_subjects": len(frame),
            "total_trials": frame["n_trials"].sum(), "mean_trials": frame["n_trials"].mean(),
            "sd_trials": frame["n_trials"].std(ddof=1), "minimum_trials": frame["n_trials"].min(),
            "maximum_trials": frame["n_trials"].max(), "mean_cnv_uv": frame["mean_cnv_uv"].mean(),
            "sd_cnv_uv": frame["mean_cnv_uv"].std(ddof=1), "rms_sme_uv": np.sqrt(np.mean(frame["sme_uv"] ** 2))})
    return participant.reset_index(), condition, pd.DataFrame(summaries)


def cnv_reliability(cnv_trials, n_splits=2000, seed=21):
    """Random split-half reliability of the probability contrast and mean CNV level."""
    subjects = sorted(cnv_trials["subject"].unique())
    if len(subjects) < 3 or n_splits < 1:
        raise ValueError("Reliability estimation requires at least three participants and one split.")
    trials = {key: group["cnv_amplitude_uv"].to_numpy(float)
              for key, group in cnv_trials.groupby(["subject", "probability"])}
    for subject in subjects:
        for probability in [80, 50]:
            if len(trials.get((subject, probability), [])) < 2:
                raise ValueError("Every participant requires at least two trials in each probability context.")
    rng = np.random.default_rng(seed)
    records = []
    for split in range(n_splits):
        first, second = np.empty((len(subjects), 2)), np.empty((len(subjects), 2))
        for i, subject in enumerate(subjects):
            for j, probability in enumerate([80, 50]):
                values = trials[(subject, probability)]
                order = rng.permutation(len(values))
                half = len(values) // 2
                first[i, j] = values[order[:half]].mean()
                second[i, j] = values[order[half:]].mean()
        measures = [
            ("probability_negativity", first[:, 1] - first[:, 0], second[:, 1] - second[:, 0]),
            ("mean_cnv_level", first.mean(axis=1), second.mean(axis=1)),
        ]
        for measure, a, b in measures:
            r = np.corrcoef(a, b)[0, 1] if np.std(a) > 0 and np.std(b) > 0 else np.nan
            sb = 2 * r / (1 + r) if np.isfinite(r) and 1 + r > 1e-12 else np.nan
            records.append({"split": split + 1, "measure": measure, "half_correlation": r,
                            "spearman_brown": sb})
    estimates = pd.DataFrame(records)
    rows = []
    for measure, group in estimates.groupby("measure", sort=False):
        valid = group["spearman_brown"].dropna()
        lower, upper = np.percentile(valid, [2.5, 97.5]) if len(valid) else [np.nan, np.nan]
        rows.append({"measure": measure, "n_subjects": len(subjects), "n_splits": n_splits,
            "n_valid_splits": len(valid), "n_undefined_splits": n_splits - len(valid), "seed": seed,
            "mean_spearman_brown": valid.mean(), "split_percentile_2_5": lower,
            "split_percentile_97_5": upper,
            "status": "ok" if len(valid) == n_splits else "undefined_estimates_omitted"})
    return pd.DataFrame(rows), estimates

## CNV predictors and mixed models

`cnv_w_z` is each trial's CNV amplitude minus that participant's mean, divided by the pooled sample SD of those deviations across all paired trials. Centering is across both probability contexts. `cnv_b_z` standardizes the participant means repeated once per paired trial, so its mean and SD are weighted by paired-trial counts. Both predictors are constructed once, before model-specific removal of missing participant scores.

| Model | Fixed effects | Correlated participant random effects |
|---|---|---|
| M3 | `CNV ~ Probability` | Intercept + Probability |
| M4 | `CNV ~ Probability * Self + Age` | Intercept + Probability |
| M5 | `P3a ~ Validity * Probability * CNV_w + CNV_b + Age` | Intercept + Validity * Probability |
| M6 | `P3a ~ Validity * Probability * CNV_w * Self + CNV_b + Age` | Intercept + Validity * Probability |

M6 additionally includes three independent participant variance components: `CNV_w`, `Validity × CNV_w`, and `Validity × Probability × CNV_w`. These components are independent of one another and of the correlated intercept/slopes.

Probability is coded −0.5 for 50% and +0.5 for 80%; validity is coded −0.5 for valid and +0.5 for invalid targets. Amplitudes are in microvolts. Models use REML, the Powell optimizer, and a maximum of 5,000 iterations. Fixed-effect tests use the normal approximation. These CNV analyses are exploratory; p-values are unadjusted.

**Sign and units.** Increasing `cnv_w_z` means less negative CNV. Coefficients involving this predictor are per one pooled within-participant SD. For interpretation in terms of increasing CNV negativity, the coupling contrast is multiplied by −1 and its confidence limits are reversed. Both conventions are exported.

In [5]:
def prepare_cnv_model_data(cnv_trials, pairs, participant_scores):
    """Attach cohort-wide scores and define signed CNV predictors once."""
    required_scores = ["subject", "self_z", "age_z"]
    missing_scores = set(required_scores) - set(participant_scores.columns)
    if missing_scores:
        raise ValueError(f"Participant scores are missing columns: {sorted(missing_scores)}")
    scores = participant_scores[required_scores].copy()
    if scores["subject"].duplicated().any():
        raise ValueError("Participant scores must contain exactly one row per subject.")

    def attach_scores(trials, label):
        overlap = {"self_z", "age_z"} & set(trials.columns)
        if overlap:
            raise ValueError(f"{label} already contains participant scores: {sorted(overlap)}")
        result = trials.merge(scores, on="subject", how="left", validate="many_to_one", indicator=True)
        absent = result.loc[result["_merge"] != "both", "subject"].unique().tolist()
        if absent:
            raise ValueError(f"{label} subjects absent from participant scores: {absent}")
        result = result.drop(columns="_merge")
        result["probability_c"] = result["probability"].map({50: -0.5, 80: 0.5})
        if result["probability_c"].isna().any():
            raise ValueError(f"{label} probability values must be 50 or 80.")
        if "validity" in result:
            result["validity_c"] = result["validity"].map({"valid": -0.5, "invalid": 0.5})
            if result["validity_c"].isna().any():
                raise ValueError(f"{label} validity values must be 'valid' or 'invalid'.")
        return result

    cnv_data = attach_scores(cnv_trials, "CNV trials")
    pair_data = attach_scores(pairs, "Paired trials")
    for data, column, label in [
        (cnv_data, "cnv_amplitude_uv", "CNV trials"),
        (pair_data, "cnv_amplitude_uv", "Paired trials"),
        (pair_data, "erp_amplitude_uv", "Paired trials"),
    ]:
        if not np.isfinite(data[column].to_numpy(dtype=float)).all():
            raise ValueError(f"{label} contains missing or nonfinite {column}.")
    if len(pair_data) < 2:
        raise ValueError("At least two paired trials are required for CNV scaling.")
    pair_data["participant_cnv_mean_uv"] = pair_data.groupby("subject")["cnv_amplitude_uv"].transform("mean")
    pair_data["cnv_within_uv"] = pair_data["cnv_amplitude_uv"] - pair_data["participant_cnv_mean_uv"]
    within_sd = float(pair_data["cnv_within_uv"].std(ddof=1))
    repeated_means = pair_data["participant_cnv_mean_uv"]
    between_mean = float(repeated_means.mean())
    between_sd = float(repeated_means.std(ddof=1))
    if not np.isfinite(within_sd) or within_sd <= 0:
        raise ValueError("The pooled within-participant CNV SD must be positive.")
    if not np.isfinite(between_sd) or between_sd <= 0:
        raise ValueError("The SD of repeated participant CNV means must be positive.")
    pair_data["cnv_w_z"] = pair_data["cnv_within_uv"] / within_sd
    pair_data["cnv_b_z"] = (repeated_means - between_mean) / between_sd
    pair_data["validity_cnv_w_z"] = pair_data["validity_c"] * pair_data["cnv_w_z"]
    pair_data["validity_probability_cnv_w_z"] = (
        pair_data["validity_c"] * pair_data["probability_c"] * pair_data["cnv_w_z"]
    )
    scaling_table = pd.DataFrame([
        {"quantity": "within_participant_cnv_pooled_sd_uv", "value": within_sd,
         "definition": "Sample SD of participant-centered CNV across all paired trials; ddof = 1"},
        {"quantity": "between_participant_cnv_trial_weighted_mean_uv", "value": between_mean,
         "definition": "Mean of participant CNV means repeated once per paired trial"},
        {"quantity": "between_participant_cnv_trial_weighted_sd_uv", "value": between_sd,
         "definition": "Sample SD of participant CNV means repeated once per paired trial; ddof = 1"},
        {"quantity": "scaling_n_subjects", "value": pair_data["subject"].nunique(),
         "definition": "All participants with paired trials before model-specific complete-case selection"},
        {"quantity": "scaling_n_trials", "value": len(pair_data),
         "definition": "All paired trials before model-specific complete-case selection"},
    ])
    return cnv_data, pair_data, scaling_table


def cnv_fixed_effect_table(model, model_id):
    """Return normal-theory summaries of fixed effects only."""
    terms = model.fe_params.index
    intervals = model.conf_int().loc[terms]
    return pd.DataFrame({
        "model": model_id,
        "term": terms,
        "b": model.fe_params.to_numpy(dtype=float),
        "SE": model.bse.loc[terms].to_numpy(dtype=float),
        "z": model.tvalues.loc[terms].to_numpy(dtype=float),
        "p": model.pvalues.loc[terms].to_numpy(dtype=float),
        "ci_lower": intervals.iloc[:, 0].to_numpy(dtype=float),
        "ci_upper": intervals.iloc[:, 1].to_numpy(dtype=float),
    })


def fit_cnv_mixed_model(data, model_id, formula, required_columns, re_formula, vc_formula=None):
    """Fit one REML model and retain its diagnostics."""
    cohort = data.dropna(subset=required_columns).copy()
    numeric_required = [column for column in required_columns if column != "subject"]
    if not np.isfinite(cohort[numeric_required].to_numpy(dtype=float)).all():
        raise ValueError(f"{model_id} contains nonfinite model variables.")
    if cohort["subject"].nunique() < 3:
        raise ValueError(f"{model_id} requires at least three complete-case participants.")
    if cohort["probability_c"].nunique() != 2:
        raise ValueError(f"{model_id} requires both probability contexts.")
    if "validity_c" in required_columns and cohort["validity_c"].nunique() != 2:
        raise ValueError(f"{model_id} requires valid and invalid target trials.")
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model = smf.mixedlm(
            formula, cohort, groups=cohort["subject"],
            re_formula=re_formula, vc_formula=vc_formula,
        ).fit(reml=True, method="powell", maxiter=5000)
    fixed = cnv_fixed_effect_table(model, model_id)
    warning_rows = [
        {"model": model_id, "category": warning.category.__name__, "message": str(warning.message)}
        for warning in caught
    ]
    covariance = model.cov_re
    covariance_rows = [
        {"model": model_id, "random_effect_row": row, "random_effect_column": column,
         "covariance": float(covariance.loc[row, column])}
        for row in covariance.index for column in covariance.columns
    ]
    vc_names = model.model.exog_vc.names if model.model.exog_vc is not None else []
    variance_rows = [
        {"model": model_id, "component": name, "variance": float(value),
         "SD": float(np.sqrt(value)) if value >= 0 else np.nan}
        for name, value in zip(vc_names, model.vcomp)
    ]
    probability_variance = float(covariance.loc["probability_c", "probability_c"])
    diagnostic = {
        "model": model_id, "formula": formula, "random_effect_formula": re_formula,
        "variance_component_formula": str(vc_formula or {}),
        "estimation": "REML", "optimizer": "Powell", "maxiter": 5000,
        "converged": bool(model.converged),
        "n_subjects": cohort["subject"].nunique(), "n_trials": len(cohort),
        "excluded_subjects_missing_model_variables": data["subject"].nunique() - cohort["subject"].nunique(),
        "excluded_trials_missing_model_variables": len(data) - len(cohort),
        "log_likelihood": float(model.llf), "residual_variance_uv2": float(model.scale),
        "minimum_random_covariance_eigenvalue": float(np.linalg.eigvalsh(covariance).min()),
        "random_probability_slope_sd_uv": float(np.sqrt(probability_variance)) if probability_variance >= 0 else np.nan,
        "n_warnings": len(warning_rows),
    }
    return model, fixed, diagnostic, warning_rows, covariance_rows, variance_rows, cohort


def run_cnv_models(cnv_data, pair_data):
    """Fit M3–M6 and return model objects, publication tables, and cohorts."""
    pair_required = ["subject", "erp_amplitude_uv", "validity_c", "probability_c", "cnv_w_z", "cnv_b_z", "age_z"]
    independent_slopes = {
        "cnv_within": "0 + cnv_w_z",
        "cnv_within_by_validity": "0 + validity_cnv_w_z",
        "cnv_within_by_probability_by_validity": "0 + validity_probability_cnv_w_z",
    }
    specifications = [
        ("M3", cnv_data, "cnv_amplitude_uv ~ probability_c",
         ["subject", "cnv_amplitude_uv", "probability_c"], "~ probability_c", None),
        ("M4", cnv_data, "cnv_amplitude_uv ~ probability_c * self_z + age_z",
         ["subject", "cnv_amplitude_uv", "probability_c", "self_z", "age_z"], "~ probability_c", None),
        ("M5", pair_data, "erp_amplitude_uv ~ validity_c * probability_c * cnv_w_z + cnv_b_z + age_z",
         pair_required, "~ validity_c * probability_c", None),
        ("M6", pair_data, "erp_amplitude_uv ~ validity_c * probability_c * cnv_w_z * self_z + cnv_b_z + age_z",
         pair_required + ["self_z"], "~ validity_c * probability_c", independent_slopes),
    ]
    models, cohorts = {}, {}
    fixed_tables, diagnostics, warning_rows, covariance_rows, variance_rows = [], [], [], [], []
    for model_id, data, formula, required, random_formula, variance_formula in specifications:
        print(f"Fitting {model_id}...")
        fitted = fit_cnv_mixed_model(data, model_id, formula, required, random_formula, variance_formula)
        model, fixed, diagnostic, model_warnings, covariance, variances, cohort = fitted
        models[model_id] = model
        cohorts[model_id] = cohort
        fixed_tables.append(fixed)
        diagnostics.append(diagnostic)
        warning_rows.extend(model_warnings)
        covariance_rows.extend(covariance)
        variance_rows.extend(variances)
    all_fixed = pd.concat(fixed_tables, ignore_index=True)
    table6_terms = {
        "M3": None, "M4": None,
        "M5": ["Intercept", "validity_c:cnv_w_z", "probability_c:cnv_w_z", "validity_c:probability_c:cnv_w_z"],
        "M6": ["Intercept", "validity_c:cnv_w_z:self_z", "probability_c:cnv_w_z:self_z", "validity_c:probability_c:cnv_w_z:self_z"],
    }
    table6_parts = []
    for model_id, terms in table6_terms.items():
        frame = all_fixed.loc[all_fixed["model"] == model_id].copy()
        if terms is not None:
            frame = frame.set_index("term").loc[terms].reset_index()
        frame["n_subjects"] = cohorts[model_id]["subject"].nunique()
        frame["n_trials"] = len(cohorts[model_id])
        table6_parts.append(frame)
    cohort_counts = []
    for model_id, cohort in cohorts.items():
        group_columns = ["subject", "probability"]
        if model_id in {"M5", "M6"}:
            group_columns.append("validity")
        counts = cohort.groupby(group_columns).size().rename("n_trials").reset_index()
        counts.insert(0, "model", model_id)
        cohort_counts.append(counts)
    tables = {
        "Table6_CNV_models": pd.concat(table6_parts, ignore_index=True),
        "All_fixed_effects": all_fixed,
        "Model_QC": pd.DataFrame(diagnostics),
        "Model_warnings": pd.DataFrame(warning_rows, columns=["model", "category", "message"]),
        "Random_covariances": pd.DataFrame(covariance_rows),
        "Variance_components": pd.DataFrame(variance_rows, columns=["model", "component", "variance", "SD"]),
        "Model_cohort_counts": pd.concat(cohort_counts, ignore_index=True),
    }
    return models, tables, cohorts


def representative_self_scores(m6_cohort):
    """Use participant-level tertile medians in the M6 complete-case cohort."""
    if m6_cohort.groupby("subject")["self_z"].nunique(dropna=False).gt(1).any():
        raise ValueError("Self-functioning scores must be constant within participants.")
    subject_scores = m6_cohort.groupby("subject")["self_z"].first().dropna()
    levels = ["Low", "Medium", "High"]
    try:
        tertiles, boundaries = pd.qcut(subject_scores, 3, labels=levels, retbins=True)
    except ValueError as exc:
        raise ValueError("Distinct self-score tertile boundaries are required for Figure 5.") from exc
    assignments = pd.DataFrame({"self_z": subject_scores, "tertile": tertiles}).reset_index()
    summary_rows = []
    for index, level in enumerate(levels):
        values = subject_scores.loc[tertiles == level]
        if values.empty:
            raise ValueError(f"Self-score tertile {level} contains no participants.")
        summary_rows.append({
            "tertile": level, "self_z_median": float(values.median()), "n_subjects": len(values),
            "lower_boundary": float(boundaries[index]), "upper_boundary": float(boundaries[index + 1]),
        })
    return assignments, pd.DataFrame(summary_rows)


def coupling_contrasts(m6, tertile_summary):
    """Estimate CNV–validity slopes in each context and both sign conventions."""
    terms = [
        "validity_c:cnv_w_z", "validity_c:probability_c:cnv_w_z",
        "validity_c:cnv_w_z:self_z", "validity_c:probability_c:cnv_w_z:self_z",
    ]
    coefficients = m6.fe_params.loc[terms].to_numpy(dtype=float)
    covariance = m6.cov_params().loc[terms, terms].to_numpy(dtype=float)
    rows = []
    for row in tertile_summary.itertuples(index=False):
        self_score = float(row.self_z_median)
        for probability, context in [(80, 0.5), (50, -0.5)]:
            weights = np.array([1.0, context, self_score, context * self_score])
            estimate = float(weights @ coefficients)
            variance = float(weights @ covariance @ weights)
            if not np.isfinite(variance) or variance <= 0:
                raise ValueError("A coupling contrast has a nonpositive or nonfinite estimated variance.")
            standard_error = float(np.sqrt(variance))
            for direction, sign in [("CNV amplitude increase", 1), ("CNV negativity increase", -1)]:
                signed_estimate = sign * estimate
                z = signed_estimate / standard_error
                rows.append({
                    "model": "M6", "tertile": row.tertile, "self_z": self_score,
                    "probability": probability, "direction": direction,
                    "change_in_cnv_w_z": sign, "b_uv_per_cnv_sd": signed_estimate,
                    "SE": standard_error, "z": z, "p": float(2 * stats.norm.sf(abs(z))),
                    "ci_lower": signed_estimate - 1.96 * standard_error,
                    "ci_upper": signed_estimate + 1.96 * standard_error,
                })
    return pd.DataFrame(rows)

## Figures 4 and 5

Figure 4 shows equally weighted participant averages at Fz, with pointwise 95% t-based confidence intervals across participants. The topographies average the 350–1,000 ms window for each participant and then average participants, using a common symmetric color scale. The stored electrode coordinates are used. Evoked trial counts and Fz window means must agree with the numerical CNV features.

Figure 5 shows M6 fixed-effect predictions of the invalid-minus-valid P3a response across signed within-participant CNV deviations (−2 to +2 SD). Negative x values indicate more negative CNV. Self scores are the medians of three participant-level quantile groups within the M6 cohort. Their exact values and group sizes are calculated from the data. The lines are model estimates without uncertainty bands; the separate coupling table provides confidence intervals for their slopes.

In [6]:
def load_cnv_averages(evoked_dir, subjects):
    """Read pooled cue averages for both probability contexts."""
    subjects = list(subjects)
    if not subjects or len(subjects) != len(set(subjects)):
        raise ValueError("Provide a nonempty list of unique participant identifiers.")
    averages, reference = {}, None
    expected_comments = {"probability_50": 50, "probability_80": 80}
    for subject in subjects:
        path = Path(evoked_dir) / f"{subject}_cnv-ave.fif"
        evokeds = mne.read_evokeds(path, baseline=None, verbose="ERROR")
        comments = [evoked.comment for evoked in evokeds]
        if len(comments) != 2 or set(comments) != set(expected_comments):
            raise ValueError(f"{subject}: expected one CNV average for each probability context.")
        averages[subject] = {}
        for evoked in evokeds:
            probability = expected_comments[evoked.comment]
            eeg = evoked.copy().pick("eeg")
            if CNV_CHANNEL not in eeg.ch_names:
                raise ValueError(f"{subject}: {CNV_CHANNEL} is missing from the EEG channels.")
            if not np.isfinite(eeg.data).all():
                raise ValueError(f"{subject}: the CNV averages contain nonfinite amplitudes.")
            locations = np.array([channel["loc"][:3] for channel in eeg.info["chs"]])
            if (eeg.get_montage() is None or not np.isfinite(locations).all()
                    or np.any(np.all(np.isclose(locations, 0), axis=1))):
                raise ValueError(f"{subject}: recorded electrode coordinates are required for the topographies.")
            if reference is None:
                reference = eeg
            elif (eeg.ch_names != reference.ch_names
                    or eeg.data.shape[1] != reference.data.shape[1]
                    or eeg.first != reference.first
                    or eeg.info["sfreq"] != reference.info["sfreq"]):
                raise ValueError("CNV averages must have matching EEG channels and sample grids.")
            averages[subject][probability] = eeg
    return averages


def _cnv_plot_style():
    return {"font.family": "DejaVu Sans", "font.size": 9,
            "axes.labelsize": 10, "xtick.labelsize": 8, "ytick.labelsize": 9,
            "legend.fontsize": 8, "axes.linewidth": .8,
            "pdf.fonttype": 42, "ps.fonttype": 42}


def _cnv_despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", length=3, width=.8)


def figure4(averages, cnv_trials):
    """Plot equal-participant CNV waveforms and window-averaged topographies."""
    required = ["subject", "probability", "cnv_amplitude_uv"]
    if not set(required).issubset(cnv_trials.columns):
        raise ValueError(f"CNV trials must contain {required}.")
    frame = cnv_trials[required].copy()
    if (frame.isna().any().any()
            or not np.isfinite(frame["cnv_amplitude_uv"].to_numpy(dtype=float)).all()
            or not frame["probability"].isin([50, 80]).all()):
        raise ValueError("Figure 4 requires finite CNV amplitudes and valid probability labels.")
    subjects = sorted(frame["subject"].unique())
    if set(subjects) != set(averages) or len(subjects) < 2:
        raise ValueError("Figure 4 requires matching participants in both inputs and N >= 2.")
    reference = averages[subjects[0]][50]
    sample_times = (np.arange(reference.data.shape[1]) + reference.first) / reference.info["sfreq"]
    window = (sample_times >= CNV_WINDOW[0]) & (sample_times <= CNV_WINDOW[1])
    if not window.any():
        raise ValueError("The CNV measurement window contains no samples.")
    trial_summaries = frame.groupby(["subject", "probability"])["cnv_amplitude_uv"].agg(["mean", "size"])
    if len(trial_summaries) != len(subjects) * 2:
        raise ValueError("Each participant must have CNV trials in both probability contexts.")
    waves, topographies, qc_rows = {}, {}, []
    for probability in [50, 80]:
        wave_rows, topo_rows = [], []
        for subject in subjects:
            evoked = averages[subject][probability]
            if (evoked.ch_names != reference.ch_names
                    or evoked.data.shape != reference.data.shape
                    or evoked.first != reference.first
                    or evoked.info["sfreq"] != reference.info["sfreq"]):
                raise ValueError("CNV averages must have matching EEG channels and sample grids.")
            trace = evoked.get_data(picks=[CNV_CHANNEL])[0] * 1e6
            measured = float(trace[window].mean())
            expected = float(trial_summaries.loc[(subject, probability), "mean"])
            n_trials = int(trial_summaries.loc[(subject, probability), "size"])
            agrees = bool(np.isclose(measured, expected, rtol=1e-5, atol=1e-5))
            count_agrees = bool(np.isclose(evoked.nave, n_trials, rtol=0, atol=1e-8))
            qc_rows.append({"subject": subject, "probability": probability,
                            "n_trials": n_trials, "evoked_nave": evoked.nave,
                            "trial_mean_uv": expected, "evoked_mean_uv": measured,
                            "difference_uv": measured - expected,
                            "amplitude_agrees": agrees, "count_agrees": count_agrees})
            if not agrees or not count_agrees:
                raise ValueError(
                    f"{subject}, {probability}%: CNV averages disagree with the trial features. "
                    "Use averages and trial features from the same preprocessing run."
                )
            wave_rows.append(trace)
            topo_rows.append(evoked.data[:, window].mean(axis=1) * 1e6)
        waves[probability] = np.asarray(wave_rows)
        topographies[probability] = np.asarray(topo_rows)
    waveform_rows, topo_rows = [], []
    summary = {}
    n_subjects = len(subjects)
    t_critical = stats.t.ppf(.975, n_subjects - 1)
    for probability in [50, 80]:
        mean_wave = waves[probability].mean(axis=0)
        interval = t_critical * stats.sem(waves[probability], axis=0, ddof=1)
        lower, upper = mean_wave - interval, mean_wave + interval
        summary[probability] = (mean_wave, lower, upper)
        for t, mean, lo, hi in zip(sample_times, mean_wave, lower, upper):
            waveform_rows.append({"probability": probability, "time_s": t,
                                  "mean_uv": mean, "ci_lower_uv": lo,
                                  "ci_upper_uv": hi, "n_subjects": n_subjects})
        topo_mean = topographies[probability].mean(axis=0)
        for channel, mean in zip(reference.ch_names, topo_mean):
            topo_rows.append({"probability": probability, "channel": channel,
                              "mean_uv": mean, "n_subjects": n_subjects,
                              "window_first_s": sample_times[window][0],
                              "window_last_s": sample_times[window][-1],
                              "window_n_samples": int(window.sum())})
    topo_means = {p: topographies[p].mean(axis=0) for p in [50, 80]}
    vmax = float(np.max(np.abs(np.concatenate(list(topo_means.values())))))
    if vmax == 0:
        vmax = 1.0
    lower_all = np.concatenate([summary[p][1] for p in [50, 80]])
    upper_all = np.concatenate([summary[p][2] for p in [50, 80]])
    ymin, ymax = lower_all.min(), upper_all.max()
    yrange = ymax - ymin if ymax > ymin else 1.0
    with plt.rc_context(_cnv_plot_style()):
        fig = plt.figure(figsize=(9.2, 3.2), facecolor="white")
        grid = fig.add_gridspec(1, 4, width_ratios=[2.8, 1.1, 1.1, .10], wspace=.35,
                               left=.07, right=.92, bottom=.18, top=.84)
        ax_wave = fig.add_subplot(grid[0, 0])
        topo_axes = [fig.add_subplot(grid[0, 1]), fig.add_subplot(grid[0, 2])]
        cax = fig.add_subplot(grid[0, 3])
        ax_wave.axvspan(CNV_WINDOW[0]*1000, CNV_WINDOW[1]*1000, color=".96", zorder=0)
        for probability, color, shade, line in [(50, ".45", ".93", "--"), (80, "black", ".82", "-")]:
            mean, lower, upper = summary[probability]
            ax_wave.fill_between(sample_times*1000, lower, upper, color=shade, linewidth=0)
            ax_wave.plot(sample_times*1000, mean, color=color, linestyle=line, linewidth=1.5,
                         label=f"{probability}%", zorder=4)
        ax_wave.axhline(0, color=".65", linewidth=.6, linestyle=":")
        ax_wave.axvline(0, color=".65", linewidth=.6, linestyle=":")
        ax_wave.set(xlim=(sample_times[0]*1000, sample_times[-1]*1000),
                    ylim=(ymin-.05*yrange, ymax+.08*yrange),
                    xlabel="Time from cue (ms)", ylabel="Amplitude (µV)")
        _cnv_despine(ax_wave)
        handles, labels = ax_wave.get_legend_handles_labels()
        ax_wave.legend(handles[::-1], labels[::-1], frameon=False, ncol=2,
                       loc="lower left", bbox_to_anchor=(0, 1.02), borderaxespad=0)
        ax_wave.text(-.14, 1.04, "A", transform=ax_wave.transAxes,
                     fontsize=12, fontweight="bold", va="bottom")
        for ax, probability, panel in zip(topo_axes, [50, 80], ["B", "C"]):
            im, _ = mne.viz.plot_topomap(topo_means[probability], reference.info, axes=ax,
                                         show=False, cmap="RdBu_r", vlim=(-vmax, vmax),
                                         contours=6, sensors=True)
            ax.set_title(f"{probability}%", fontsize=10, pad=5)
            ax.text(-.18, 1.04, panel, transform=ax.transAxes,
                    fontsize=12, fontweight="bold", va="bottom")
        fig.colorbar(im, cax=cax).set_label("µV")
    return fig, pd.DataFrame(waveform_rows), pd.DataFrame(topo_rows), pd.DataFrame(qc_rows)


def _validity_prediction(fe, probability_c, self_z, cnv_w_z):
    """M6 invalid-minus-valid prediction at specified moderator values."""
    # Resolve interaction terms independent of the factor order in the formula.
    def coefficient(*factors):
        matches = [term for term in fe.index if set(term.split(":")) == set(factors)]
        if len(matches) != 1:
            raise ValueError(f"Expected exactly one model term for {':'.join(factors)}.")
        return float(fe.loc[matches[0]])
    intercept = (coefficient("validity_c")
                 + coefficient("validity_c", "probability_c") * probability_c
                 + coefficient("validity_c", "self_z") * self_z
                 + coefficient("validity_c", "probability_c", "self_z") * probability_c * self_z)
    slope = (coefficient("validity_c", "cnv_w_z")
             + coefficient("validity_c", "probability_c", "cnv_w_z") * probability_c
             + coefficient("validity_c", "cnv_w_z", "self_z") * self_z
             + coefficient("validity_c", "probability_c", "cnv_w_z", "self_z") * probability_c * self_z)
    return intercept + slope * np.asarray(cnv_w_z, dtype=float)


def figure5(m6, tertile_summary):
    """Plot M6 validity effects across within-person CNV and self impairment."""
    levels = ["Low", "Medium", "High"]
    required = ["tertile", "self_z_median", "n_subjects", "lower_boundary", "upper_boundary"]
    if not set(required).issubset(tertile_summary.columns):
        raise ValueError(f"Tertile summary must contain {required}.")
    frame = tertile_summary.set_index("tertile", verify_integrity=True)
    if set(frame.index) != set(levels) or not np.isfinite(frame["self_z_median"]).all():
        raise ValueError("Figure 5 requires three finite tertile medians.")
    x = np.linspace(-2, 2, 201)
    rows, predictions = [], {}
    for probability, centered in [(50, -.5), (80, .5)]:
        for level in levels:
            self_value = float(frame.loc[level, "self_z_median"])
            y = _validity_prediction(m6.fe_params, centered, self_value, x)
            if not np.isfinite(y).all():
                raise ValueError("M6 predictions contain nonfinite values.")
            predictions[(probability, level)] = y
            for x_value, y_value in zip(x, y):
                rows.append({"probability": probability, "probability_c": centered,
                             "tertile": level, "self_z_median": self_value,
                             "tertile_n_subjects": int(frame.loc[level, "n_subjects"]),
                             "self_z_lower_boundary": frame.loc[level, "lower_boundary"],
                             "self_z_upper_boundary": frame.loc[level, "upper_boundary"],
                             "cnv_w_z": x_value, "validity_effect_uv": y_value})
    all_y = np.concatenate(list(predictions.values()))
    ymin, ymax = min(0., float(all_y.min())), max(0., float(all_y.max()))
    pad = .1 * max(ymax-ymin, .5)
    colors = {"Low": ".10", "Medium": ".40", "High": ".62"}
    styles = {"Low": "-", "Medium": (0, (4, 2)), "High": (0, (1, 2))}
    with plt.rc_context(_cnv_plot_style()):
        fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.8), sharey=True, facecolor="white")
        fig.subplots_adjust(left=.12, right=.98, bottom=.30, top=.88, wspace=.24)
        for ax, probability, panel in zip(axes, [50, 80], ["A", "B"]):
            for level in levels:
                ax.plot(x, predictions[(probability, level)], color=colors[level], linestyle=styles[level],
                        linewidth=1.9, label=level, solid_capstyle="round", dash_capstyle="round")
            ax.axhline(0, color=".82", linewidth=.7, zorder=0)
            ax.set_title(panel, loc="left", fontsize=11, fontweight="bold", pad=10)
            ax.set_title(f"{probability}%", loc="center", fontsize=10, fontweight="normal", pad=10)
            ax.set(xlim=(-2.05, 2.05), ylim=(ymin-pad, ymax+pad),
                   xlabel="Within-person CNV (SD units)", xticks=[-2, -1, 0, 1, 2])
            ax.locator_params(axis="y", nbins=5)
            ax.minorticks_off()
            _cnv_despine(ax)
        axes[0].set_ylabel("Validity effect\n(invalid − valid, µV)", labelpad=9)
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(.55, .025), ncol=3,
                   frameon=False, title="Self impairment (tertile medians)",
                   handlelength=3, handletextpad=.7, columnspacing=2.2)
    return fig, pd.DataFrame(rows)

## Run the analyses

In [7]:
result_tables = None
models, cohorts, figures = {}, {}, {}
if RUN_ANALYSES:
    cnv_trials, paired_trials, participant_scores, cnv_qc, input_files = load_cnv_inputs()
    participant_cnv, condition_cnv, cnv_summary = summarize_cnv_trials(cnv_trials)
    reliability, split_estimates = cnv_reliability(cnv_trials, N_RELIABILITY_SPLITS, RELIABILITY_SEED)
    cnv_data, pair_data, scaling = prepare_cnv_model_data(cnv_trials, paired_trials, participant_scores)
    models, model_tables, cohorts = run_cnv_models(cnv_data, pair_data)
    tertile_assignments, tertile_summary = representative_self_scores(cohorts["M6"])
    coupling = coupling_contrasts(models["M6"], tertile_summary)
    result_tables = {
        "Participant_CNV": participant_cnv, "Condition_CNV": condition_cnv, "CNV_summary": cnv_summary,
        "CNV_reliability": reliability, "Reliability_splits": split_estimates,
        "CNV_predictor_scaling": scaling, "Paired_model_data": pair_data,
        "Participant_scores": participant_scores, "CNV_inclusion_QC": cnv_qc,
        "Self_tertile_assignments": tertile_assignments, "Self_tertile_summary": tertile_summary,
        "Coupling_simple_slopes": coupling, "Software_versions": software_versions,
        "Input_files": input_files, **model_tables,
    }
    if MAKE_FIGURES:
        averages = load_cnv_averages(EVOKED_DIR, sorted(cnv_trials["subject"].unique()))
        figures["Figure4_CNV"], waves, topographies, evoked_qc = figure4(averages, cnv_trials)
        figures["Figure5_CNV_P3a_coupling"], predictions = figure5(models["M6"], tertile_summary)
        result_tables.update({"Figure4_waveforms": waves, "Figure4_topographies": topographies,
                              "CNV_evoked_QC": evoked_qc, "Figure5_predictions": predictions})
        plt.show()
    settings = {
        "cnv_channel": CNV_CHANNEL, "cnv_window_s": CNV_WINDOW,
        "minimum_cnv_trials_per_probability": MIN_CNV_TRIALS_PER_CONTEXT,
        "cnv_saccade_rule": "detected; target-relative latency >= 0 ms; any direction",
        "participant_score_source": "Participant_scores from notebook 03; no restandardization",
        "within_cnv_scaling": "participant-centered across contexts / pooled paired-trial SD; ddof=1",
        "between_cnv_scaling": "z-score of participant means repeated on paired rows; ddof=1",
        "scaling_population": "all paired trials before model-specific complete-case selection",
        "reliability_splits": N_RELIABILITY_SPLITS, "reliability_seed": RELIABILITY_SEED,
        "reliability_aggregation": "mean of split-specific Spearman-Brown estimates",
        "odd_trial_split_rule": "first half=floor(n/2); second half=remaining trials",
        "figure4_intervals": "pointwise 95% t intervals across participant means",
        "figure5_self_scores": "tertile medians among unique M6 participants",
        "p_value_adjustment": "none; exploratory CNV analyses",
    }
    result_tables["Analysis_settings"] = pd.DataFrame(
        [(key, str(value)) for key, value in settings.items()], columns=["setting", "value"])
else:
    display(Markdown("Study analysis is disabled. Configure the paths and set `RUN_ANALYSES = True`."))

Study analysis is disabled. Configure the paths and set `RUN_ANALYSES = True`.

## Inspect the results

`Table6_CNV_models` contains all M3/M4 fixed effects and the selected M5/M6 effects reported in Table 6. `All_fixed_effects` retains every fixed-effect estimate. Check model-specific participant/trial counts and convergence before interpretation. A returned coefficient table alone does not establish that the fit is satisfactory.

In `Coupling_simple_slopes`, the rows labeled `CNV negativity increase` express the change in the P3a validity effect for a one-SD increase in CNV negativity. Positive values indicate a larger invalid-minus-valid response.

In [8]:
if result_tables is not None:
    for sheet in ["CNV_summary", "CNV_reliability", "CNV_predictor_scaling", "Model_QC", "Table6_CNV_models", "Self_tertile_summary"]:
        display(Markdown(f"### {sheet.replace('_', ' ')}"))
        display(result_tables[sheet])
    if not result_tables["Model_warnings"].empty:
        display(Markdown("### Model warnings"))
        display(result_tables["Model_warnings"])
    display(Markdown("### Coupling per one-SD increase in CNV negativity"))
    display(result_tables["Coupling_simple_slopes"].loc[
        result_tables["Coupling_simple_slopes"]["direction"].eq("CNV negativity increase")])

## Export

`cnv_statistics.xlsx` contains the model results, sample counts, reliability estimates, scaling constants, participant-level scores, paired-trial predictors, and numerical figure data. Figure files are exported as vector PDF and 600-dpi TIFF.

The workbook contains participant-level research data; keep the analysis outputs separate from the public code. Set `EXPORT_OUTPUTS = True` to write results. Existing files are replaced only when `OVERWRITE_OUTPUTS = True`.

In [9]:
def export_cnv_statistics(tables, figures, output_dir, overwrite=False):
    output_dir = Path(output_dir).expanduser()
    if "PATH_TO_" in str(output_dir):
        raise ValueError("Set OUTPUT_DIR before exporting.")
    workbook = output_dir / "cnv_statistics.xlsx"
    targets = [workbook] + [output_dir / f"{name}{extension}"
                            for name in figures for extension in (".pdf", ".tiff")]
    existing = [path.name for path in targets if path.exists()]
    if existing and not overwrite:
        raise FileExistsError(f"Outputs already exist: {existing}. Set OVERWRITE_OUTPUTS to replace them.")
    output_dir.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(workbook, engine="openpyxl") as writer:
        for sheet, table in tables.items():
            table.to_excel(writer, sheet_name=sheet, index=False)
    for name, figure in figures.items():
        figure.savefig(output_dir / f"{name}.pdf", bbox_inches="tight", facecolor="white")
        figure.savefig(output_dir / f"{name}.tiff", dpi=600, bbox_inches="tight", facecolor="white",
                       pil_kwargs={"compression": "tiff_lzw"})
    return pd.DataFrame({"file": [path.name for path in targets]})

if EXPORT_OUTPUTS:
    if result_tables is None:
        raise RuntimeError("Run the analysis before exporting.")
    display(export_cnv_statistics(result_tables, figures, OUTPUT_DIR, OVERWRITE_OUTPUTS))
else:
    print("Export disabled; no participant results written.")

Export disabled; no participant results written.
